In [ ]:
# ============================================================
# MEMBER 4 - AI / ML IRRIGATION PREDICTION
# ============================================================

!pip -q install pandas numpy scikit-learn joblib

import pandas as pd
import numpy as np
import joblib

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# 1. UPLOAD CSV
# ============================================================

uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("✅ CSV loaded successfully")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

display(df)


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nColumns after cleaning:")
print(df.columns.tolist())


# ============================================================
# 3. REQUIRED COLUMNS
# ============================================================

required_columns = [
    "crop",
    "soil_moisture",
    "temperature",
    "humidity",
    "rainfall",
    "soil_ph",
    "irrigation_requirement"
]

missing = [
    column for column in required_columns
    if column not in df.columns
]

if missing:
    print("❌ Missing columns:", missing)
else:
    print("✅ All required columns found!")


# ============================================================
# 4. SELECT REQUIRED COLUMNS
# ============================================================

df = df[required_columns].copy()


# ============================================================
# 5. CLEAN IRRIGATION COLUMN
# ============================================================

df["irrigation_requirement"] = (
    df["irrigation_requirement"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["irrigation_requirement"] = (
    df["irrigation_requirement"]
    .map({
        "yes": 1,
        "no": 0
    })
)

df = df.dropna(
    subset=["irrigation_requirement"]
)

print("\nIrrigation values:")
print(df["irrigation_requirement"].value_counts())


# ============================================================
# 6. INPUT (X) AND OUTPUT (Y)
# ============================================================

X = df[
    [
        "crop",
        "soil_moisture",
        "temperature",
        "humidity",
        "rainfall",
        "soil_ph"
    ]
]

y = df["irrigation_requirement"]


# ============================================================
# 7. DEFINE COLUMN TYPES
# ============================================================

numeric_columns = [
    "soil_moisture",
    "temperature",
    "humidity",
    "rainfall",
    "soil_ph"
]

categorical_columns = [
    "crop"
]


# ============================================================
# 8. PREPROCESSING
# ============================================================

numeric_processor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_processor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_processor,
            numeric_columns
        ),
        (
            "categorical",
            categorical_processor,
            categorical_columns
        )
    ]
)


# ============================================================
# 9. CREATE RANDOM FOREST MODEL
# ============================================================

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


# ============================================================
# 10. CREATE COMPLETE PIPELINE
# ============================================================

pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            model
        )
    ]
)


# ============================================================
# 11. TRAIN / TEST SPLIT
# ============================================================

# NOTE:
# Your current CSV has only 9 rows.
# This split is mainly for testing the code.
# A larger dataset is recommended for real ML training.

if len(df) >= 10:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    print("\n==============================")
    print("MODEL ACCURACY")
    print("==============================")

    print(
        "Accuracy:",
        round(accuracy * 100, 2),
        "%"
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

else:

    print(
        "\n⚠️ Dataset has only",
        len(df),
        "rows."
    )

    print(
        "Training the model using all available data."
    )

    pipeline.fit(
        X,
        y
    )


# ============================================================
# 12. TRAIN FINAL MODEL USING ALL DATA
# ============================================================

pipeline.fit(
    X,
    y
)

print("\n✅ Final AI model trained!")


# ============================================================
# 13. PREDICTION FUNCTION
# ============================================================

def predict_irrigation(
    crop,
    soil_moisture,
    temperature,
    humidity,
    rainfall,
    soil_ph
):

    new_data = pd.DataFrame({

        "crop": [crop],

        "soil_moisture": [
            soil_moisture
        ],

        "temperature": [
            temperature
        ],

        "humidity": [
            humidity
        ],

        "rainfall": [
            rainfall
        ],

        "soil_ph": [
            soil_ph
        ]

    })

    prediction = pipeline.predict(
        new_data
    )[0]

    if prediction == 1:

        irrigation = "YES"

    else:

        irrigation = "NO"

    return irrigation


# ============================================================
# 14. RECOMMENDATION FUNCTION
# ============================================================

def recommendation(
    irrigation,
    soil_moisture,
    temperature,
    rainfall
):

    reasons = []

    if soil_moisture < 30:
        reasons.append(
            "soil moisture is low"
        )

    if temperature > 32:
        reasons.append(
            "temperature is high"
        )

    if rainfall < 5:
        reasons.append(
            "rainfall is low"
        )

    if irrigation == "YES":

        if reasons:

            return (
                "Irrigation is recommended because "
                + ", ".join(reasons)
                + "."
            )

        return (
            "Irrigation is recommended "
            "based on the AI prediction."
        )

    else:

        return (
            "Irrigation is currently not "
            "required according to the AI model."
        )


# ============================================================
# 15. TEST A NEW FARM CONDITION
# ============================================================

crop = "Tomato"

soil_moisture = 27

temperature = 29

humidity = 58

rainfall = 2.5

soil_ph = 6.2


result = predict_irrigation(

    crop,
    soil_moisture,
    temperature,
    humidity,
    rainfall,
    soil_ph

)


advice = recommendation(

    result,
    soil_moisture,
    temperature,
    rainfall

)


# ============================================================
# 16. DISPLAY RESULT
# ============================================================

print("\n")
print("======================================")
print("       🌱 AI IRRIGATION RESULT")
print("======================================")

print("Crop:", crop)

print("Soil Moisture:", soil_moisture, "%")

print("Temperature:", temperature, "°C")

print("Humidity:", humidity, "%")

print("Rainfall:", rainfall, "mm")

print("Soil pH:", soil_ph)

print("--------------------------------------")

print(
    "Irrigation Required:",
    result
)

print(
    "Recommendation:",
    advice
)

print("======================================")


# ============================================================
# 17. SAVE MODEL
# ============================================================

joblib.dump(
    pipeline,
    "irrigation_model.pkl"
)

print(
    "\n✅ Model saved as irrigation_model.pkl"
)